In [1]:
import pydartdiags.obs_sequence.obs_sequence as obsq
import pandas as pd
import datetime as dt

In [2]:
def create_obs_seq(n_copies: int, n_qc: int, copie_names: list, non_qc_copie_names: list, qc_copie_names: list, n_non_qc: int):
    """Creates new obs_sequence object with column titles, header, and attributes"""
    obs_seq = obsq.ObsSequence(None)
    obs_seq.loc_mod = 'loc3d'
    obs_seq.n_copies = n_copies
    obs_seq.copie_names = copie_names
    obs_seq.non_qc_copie_names = non_qc_copie_names
    obs_seq.qc_copie_names = qc_copie_names
    obs_seq.n_non_qc = n_non_qc
    obs_seq.n_qc = n_qc
    obs_seq.df = pd.DataFrame(columns=obs_seq._column_headers())
    obs_seq.create_header(2)
    return obs_seq

In [3]:
def convert_to_dart_time(datetime: dt.datetime):
    """Converts normal date and time to DART readable days and seconds"""
    delta = datetime - dt.datetime(1601, 1, 1)
    days = delta.days
    seconds = delta.seconds
    return [days, seconds]

In [4]:
def add_obs_to_obs_seq(obs_seq, latitude: float, longitude: float, vertical: float, vert_unit: int, obs_type: str, datetime: dt.datetime, obs_err_var: float, NCEP_QC_index: int, metadata=[], external_FO=[]):
    """Adds row to created obs_seq dataframe and updates obs_seq header, attributes"""
    dart_time = convert_to_dart_time(datetime)
    new_obs = pd.DataFrame({'latitude': [float(latitude)], 'longitude': [float(longitude)], 'vertical': [vertical], 'vert_unit': [vert_unit], 'type': [obs_type], 'metadata': [metadata], 'external_FO': [external_FO], 'time' : [datetime.strftime("%H:%M:%S")], 'seconds': [dart_time[1]], 'days':[dart_time[0]], 'obs_err_var': [obs_err_var]})
    obs_seq.df = pd.concat([obs_seq.df, new_obs], ignore_index=True)
    #moves columns to fulfill list_to_obs requirements
    column_to_move = obs_seq.df.pop('obs_num')
    obs_seq.df.insert(0, 'obs_num', column_to_move)
    column_to_move = obs_seq.df.pop('linked_list')
    obs_seq.df.insert(1, 'linked_list', column_to_move)
    obs_seq.create_header_from_dataframe()
    obs_seq.update_attributes_from_df()

In [5]:
"""copie_names = ['observations', 'qc']
qc_copie_names = ['qc']
non_qc_copie_names = ['observations']"""
copie_names = []
qc_copie_names = []
non_qc_copie_names = []

In [6]:
#obs_seq = create_obs_seq(2, 1, copie_names, non_qc_copie_names, qc_copie_names, 1)
obs_seq = create_obs_seq(0, 0, copie_names, non_qc_copie_names, qc_copie_names, 0)

In [7]:
print(obs_seq.df)

Empty DataFrame
Columns: [obs_num, linked_list, longitude, latitude, vertical, vert_unit, type, metadata, external_FO, seconds, days, time, obs_err_var]
Index: []


In [8]:
obs_seq.header

['obs_sequence',
 'obs_type_definitions',
 '0',
 'num_copies: 0  num_qc: 0',
 'num_obs: 2  max_num_obs: 2',
 'first: 1 last: 2']

In [9]:
add_obs_to_obs_seq(obs_seq, 14.3457, 44.3453,5.08,'height (m)','RADIOSONDE_TEMPERATURE', dt.datetime(2025,1,1,12), 0.000001, 2.0)
add_obs_to_obs_seq(obs_seq, 14.3457, 44.3453,5.08,'height (m)','RADIOSONDE_TEMPERATURE', dt.datetime(2025,1,2,12), 0.000001, 2.0)
add_obs_to_obs_seq(obs_seq, 14.3457, 44.3453,5.08,'height (m)','RADIOSONDE_TEMPERATURE', dt.datetime(2025,1,3,12), 0.000001, 2.0)

/var/folders/5g/z6jjz7zs57dgdfld3wq9c8rm0000gp/T/ipykernel_92711/2295609208.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  obs_seq.df = pd.concat([obs_seq.df, new_obs], ignore_index=True)


In [10]:
print(obs_seq.df)

   obs_num                linked_list  longitude  latitude  vertical  \
0        1  -1          2          -1    44.3453   14.3457      5.08   
1        2  1           3          -1    44.3453   14.3457      5.08   
2        3  2           -1         -1    44.3453   14.3457      5.08   

    vert_unit                    type metadata external_FO seconds    days  \
0  height (m)  RADIOSONDE_TEMPERATURE       []          []   43200  154863   
1  height (m)  RADIOSONDE_TEMPERATURE       []          []   43200  154864   
2  height (m)  RADIOSONDE_TEMPERATURE       []          []   43200  154865   

       time  obs_err_var  
0  12:00:00     0.000001  
1  12:00:00     0.000001  
2  12:00:00     0.000001  


In [11]:
obs_seq.header

['obs_sequence',
 'obs_type_definitions',
 '1',
 '1 RADIOSONDE_TEMPERATURE',
 'num_copies: 0  num_qc: 0',
 'num_obs:          3 max_num_obs:          3',
 'first:            1 last:            3']

In [12]:
obs_seq.write_obs_seq('obs_seq.test')

[1, '-1          2          -1', 0.7739714927846415, 0.2503796985033506, 5.08, 'height (m)', 'RADIOSONDE_TEMPERATURE', [], [], 43200, 154863, '12:00:00', 1e-06]
[2, '1           3          -1', 0.7739714927846415, 0.2503796985033506, 5.08, 'height (m)', 'RADIOSONDE_TEMPERATURE', [], [], 43200, 154864, '12:00:00', 1e-06]
[3, '2           -1         -1', 0.7739714927846415, 0.2503796985033506, 5.08, 'height (m)', 'RADIOSONDE_TEMPERATURE', [], [], 43200, 154865, '12:00:00', 1e-06]


In [13]:
!cat obs_seq.test

obs_sequence
obs_type_definitions
1
1 RADIOSONDE_TEMPERATURE
num_copies: 0  num_qc: 0
num_obs:          3 max_num_obs:          3
first:            1 last:            3
OBS        1
-1          2          -1
obdef
loc3d
0.7739714927846415   0.2503796985033506   5.08   3
kind
1
43200 154863
1e-06
OBS        2
1           3          -1
obdef
loc3d
0.7739714927846415   0.2503796985033506   5.08   3
kind
1
43200 154864
1e-06
OBS        3
2           -1         -1
obdef
loc3d
0.7739714927846415   0.2503796985033506   5.08   3
kind
1
43200 154865
1e-06
